In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# Data curation

## **STEP 1**. Merge dicomtocsv_series.csv

### <span style="color:blue">**Main**</span> (determine which part of data)

In [3]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/R3Data"
file_series_p1 = "dicomtocsv_series_20250527.xlsx"
file_series_p2 = "dicomtocsv_series_20260803.xlsx"
file_series_p3 = "dicomtocsv_series_20260812.xlsx"

In [4]:
df_dicom_series_p1 = pd.read_excel(os.path.join(file_path, file_series_p1))
df_dicom_series_p2 = pd.read_excel(os.path.join(file_path, file_series_p2))
df_dicom_series_p3 = pd.read_excel(os.path.join(file_path, file_series_p3))

In [5]:
df_dicom_series_p1.shape, df_dicom_series_p2.shape, df_dicom_series_p3.shape

((114473, 7), (713, 7), (2423, 7))

In [6]:
df_tmp = pd.concat((df_dicom_series_p1, df_dicom_series_p2), axis=0)
df_all = pd.concat((df_tmp, df_dicom_series_p3), axis=0)

# df_all = pd.concat((df_dicom_series_p2, df_dicom_series_p3), axis=0)

df_all.shape

(117609, 7)

In [7]:
df_all['StudyDate'] = pd.to_datetime(df_all['StudyDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [8]:
df_all.head(5)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
0,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,1,Hologic R2 ImageChecker CAD SC,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R ML,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R XCCL,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71300000,L LM C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71300000,L MLO C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


## **STEP 2**. Group by PatientID

### <span style="color:blue">**Main**</span>

In [9]:
df_sort = df_all.sort_values(
        by=['PatientID', 'StudyDate', 'AccessionNumber'],
        ignore_index=True
    )

num_patient = df_sort["PatientID"].nunique()
print(num_patient)

5577


In [10]:
df_sort.head(5)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
0,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,1,Hologic R2 ImageChecker CAD SC,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R ML,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R XCCL,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71300000,L LM C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71300000,L MLO C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [11]:
df_step2 = df_sort.copy()

## **STEP 2.1**. Calculate Age @ Study <span style="color:red">(OPTIONAL)</span>
### <span style="color:red"> New batches: Missing patient/_demo birth_date info. 50 patients; All: 54 patients </span>

### <span style="color:darkcyan">**Function**</span>

In [12]:
from pandas import Int64Dtype
def calculate_age_at_study(df):
    """
    Calculates the patient's age in years at the time of the study 
    based on the 'PatientBirthDate' and 'StudyDate' columns.
    """
    
    # 1. Ensure date columns are in datetime format
    # The format 'YYYY-MM-DD' is used for parsing
    try:
        df['BIRTH_DATE_DT'] = pd.to_datetime(df['PatientBirthDate'], format='%Y-%m-%d')
        # Note: The StudyDate column often includes time (e.g., '2020-06-01 09:10:52').
        # We can let pandas infer the format for this one since it's cleaner.
        df['StudyDate_DT'] = pd.to_datetime(df['StudyDate'], errors='coerce') 
    except ValueError as e:
        print(f"Error parsing date format: {e}. Please check your date column formats.")
        return df

    # 2. Calculate the difference in days
    time_difference = df['StudyDate_DT'] - df['BIRTH_DATE_DT']
    
    # 3. Convert the difference into whole years (integer format)
    # The .dt.days attribute gives the number of days, which is divided by 365.25 
    # and then explicitly cast to an integer to capture only the full years elapsed.
    mask = df['BIRTH_DATE_DT'].notna() & df['StudyDate_DT'].notna()
    df.loc[mask, 'PatientAge'] = (time_difference[mask].dt.days / 365.25).astype(int)
    # df['PatientAge'] = (time_difference.dt.days / 365.25).astype('Int64')
    
    # Clean up the intermediate columns
    
    df = df.drop(columns=['BIRTH_DATE_DT', 'StudyDate_DT'])
    
    return df

### <span style="color:blue">**Main**</span>

In [13]:
demo_file_path_cancer = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/Cancer/Cleaned/patient_demo.xlsx"
demo_file_path_control = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/Control/Cleaned/patient_demo.xlsx"

demo_cancer = pd.read_excel(demo_file_path_cancer)[["PATIENT_STUDY_ID", "BIRTH_DATE"]]
demo_control = pd.read_excel(demo_file_path_control)[["PATIENT_STUDY_ID", "BIRTH_DATE"]]

In [14]:
demo =  pd.concat((demo_cancer, demo_control), axis=0)
demo = demo.drop_duplicates(subset=demo.columns.tolist(), keep = 'first').reset_index(drop = True)

In [15]:
demo.rename(columns={'PATIENT_STUDY_ID': 'PatientID', 'BIRTH_DATE': 'PatientBirthDate'}, inplace=True)

In [16]:
demo.head(3)

,PatientID,PatientBirthDate
0,4330018595,1965-07-01
1,4330029102,1974-07-01
2,4330044371,1975-07-01


In [17]:
df_birth = pd.merge(df_sort, demo, on='PatientID', how='left')

In [18]:
df_sort.shape, demo.shape, df_birth.shape

((117609, 7), (47245, 2), (117609, 8))

In [19]:
df_step2 = calculate_age_at_study(df_birth)

In [20]:
df_step2['PatientID'].nunique(), df_step2.loc[df_step2['PatientAge'].notna(), 'PatientID'].nunique(), df_step2.loc[df_step2['PatientAge'].isna(), 'PatientID'].nunique()

(5577, 5523, 54)

In [21]:
df_step2.loc[df_step2['PatientAge'].isna(), 'PatientID'].unique()

array([4333000921, 4333012455, 4333016761, 4333024138, 4333027901,
       4333031047, 4333032244, 4333042111, 4333043117, 4333055254,
       4333057495, 4333073168, 4333078808, 4333082262, 4333089213,
       4333098676, 4333322368, 4333324955, 4333339192, 4333339578,
       4333343566, 4333369221, 4333370187, 4333391549, 4333398550,
       4333402040, 4333419154, 4333447879, 4333447971, 4333455214,
       4333471743, 4333481825, 4333483028, 4333490388, 4333493561,
       4333520281, 4333520420, 4333532845, 4333535298, 4333584454,
       4333588635, 4333591910, 4333596430, 4333606098, 4333643575,
       4333646458, 4333651350, 4333659749, 4333661051, 4333667640,
       4337289159, 4337289942, 4337724180, 4337734199], dtype=int64)

In [22]:
df_step2

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath,PatientBirthDate,PatientAge
0,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,1,Hologic R2 ImageChecker CAD SC,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
1,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R ML,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
2,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R XCCL,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
3,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71300000,L LM C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
4,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71300000,L MLO C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
...,...,...,...,...,...,...,...,...,...
117604,4339959661,2019-10-28,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,62216536,71300000,R MLO C-View,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1971-07-01,48.0
117605,4339959661,2019-10-28,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,62216536,72100000,L CC Tomosynthesis Projection,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1971-07-01,48.0
117606,4339959661,2019-10-28,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,62216536,72100000,L MLO Tomosynthesis Projection,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1971-07-01,48.0
117607,4339959661,2019-10-28,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,62216536,72100000,R CC Tomosynthesis Projection,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1971-07-01,48.0


## **STEP 3.** Add tags (Study, Side, Series)

In [23]:
dicom = df_step2

In [24]:
dicom.head(3)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath,PatientBirthDate,PatientAge
0,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,1,Hologic R2 ImageChecker CAD SC,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
1,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R ML,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0
2,4330018595,2019-08-19,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,63737104,71100000,R XCCL,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...,1965-07-01,54.0


In [25]:
dicom_copy = dicom.copy()

### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [26]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [27]:
column_to_check = 'StudyDescription'
type_column = 'Study'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Side**</span> (R, L)

In [28]:
side_types = {
    "R": ["RIGHT", "RT", "R XCCL", "R MLO", "R CC", "R ML", "R SIO", "R LM"],
    "L": ["LEFT", "LT", "L XCCL", "L MLO", "L CC", "L ML", "L SIO", "L LM"],
}

In [29]:
column_to_check = 'SeriesDescription'
type_column = 'Side'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in side_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Series**</span> (DBT, IN2D, C VIEW, SECURE)

In [30]:
series_types = {
    "DBT":    ["Breast Tomosynthesis"],
    "IN2D":   ["Intelligent 2D"],
    "C VIEW": ["C-View"],
    "SECURE": ["SecurView", "CAD SC"],
}

ffdm_exact = ["R CC", "R MLO", "R ML", "L CC", "L MLO", "L ML",
              "L XCCL", "R XCCL", "R LM", "L LM",
              "R SIO", "L SIO", "RT", "LT"
              ]

In [31]:
column_to_check = 'SeriesDescription'
type_column = 'Series'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

# Exact matches for FFDM
ffdm_mask = dicom_copy[column_to_check].str.strip().str.upper().isin(
    [t.upper() for t in ffdm_exact]
)
dicom_copy.loc[ffdm_mask, type_column] = "FFDM"

### <span style="color:blue">**View**</span> (MLO, CC)

In [32]:
view_types = {

    "CC":  ["L CC", "R CC"],
    "ML":  ["L ML", "R ML"],
    "XCCL":  ["L XCCL", "R XCCL"],
    "LM":  ["L LM", "R LM"],
    "MLO": ["L MLO", "R MLO"],
    "SIO": ["R SIO", "L SIO"],
}

In [33]:
column_to_check = 'SeriesDescription'
type_column = 'View'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in view_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Reorder columns**</span>

In [34]:
dicom_copy.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'PatientBirthDate',
       'PatientAge', 'Study', 'Side', 'Series', 'View'],
      dtype='object')

In [35]:
# dicom_copy = dicom_copy[['PatientID', 
#         'AccessionNumber', 'Study', 'Side', 'Series', 'View',    
#         'StudyDate', 'StudyDescription',
#         'SeriesDescription', 'SeriesNumber', 
#         'FolderPath'
#         ]]

dicom_copy = dicom_copy[['PatientID', 'PatientBirthDate', 'PatientAge',
        'AccessionNumber', 'StudyDate',
        'Study', 'Side', 'Series', 'View',    
        'StudyDescription',
        'SeriesDescription', 'SeriesNumber', 
        'FolderPath'
        ]]

In [36]:
dicom_copy.head(3)

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,ML,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R ML,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,XCCL,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


### <span style="color:#FF6347;">**SAVE**</span> file

In [37]:
path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation"

In [ ]:
output_file = os.path.join(path,'dicom_tag' + ".xlsx")
dicom_copy.to_excel(output_file, index=False)

### <span style="color:#FF6347;">**READ**</span> file

In [ ]:
file_path = os.path.join(path,'dicom_tag' + ".xlsx")
dicom = pd.read_excel(file_path)

In [40]:
dicom[dicom["Series"]=="DBT"]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
39,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
40,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
41,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
42,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
75,4330116791,1963-07-01,56.0,61499674,2019-12-17,DIAG,L,DBT,CC,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
117417,4339601084,1972-07-01,46.0,77038224,2018-11-07,DIAG,L,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
117418,4339601084,1972-07-01,46.0,77038224,2018-11-07,DIAG,R,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
117419,4339601084,1972-07-01,46.0,77038224,2018-11-07,DIAG,R,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
117426,4339601084,1972-07-01,46.0,65758752,2019-05-16,DIAG,R,DBT,CC,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [41]:
dicom[dicom["Series"]=="DBT"]["PatientID"].unique().size

1117

In [42]:
dicom[dicom['PatientID']==4333000414]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
2089,4333000414,1960-07-01,56.0,71446941,2017-05-11,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2090,4333000414,1960-07-01,56.0,71446941,2017-05-11,DIAG,L,FFDM,CC,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,L CC,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2091,4333000414,1960-07-01,56.0,71446941,2017-05-11,DIAG,L,C VIEW,CC,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,L CC C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2092,4333000414,1960-07-01,56.0,71446941,2017-05-11,DIAG,L,FFDM,ML,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,L ML,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2093,4333000414,1960-07-01,56.0,71446941,2017-05-11,DIAG,L,C VIEW,ML,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,L ML C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2094,4333000414,1960-07-01,56.0,71446941,2017-05-11,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,SecurView Secondary Capture,74100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2095,4333000414,1960-07-01,57.0,70450443,2018-05-07,SCREEN,NaN,SECURE,NaN,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2096,4333000414,1960-07-01,57.0,70450443,2018-05-07,SCREEN,L,FFDM,CC,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,L CC,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2097,4333000414,1960-07-01,57.0,70450443,2018-05-07,SCREEN,L,C VIEW,CC,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,L CC C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2098,4333000414,1960-07-01,57.0,70450443,2018-05-07,SCREEN,L,FFDM,MLO,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,L MLO,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [43]:
dicom[dicom['PatientID']==4330018595]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,ML,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R ML,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,XCCL,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,LM,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L LM C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
5,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,SIO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L SIO C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
6,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,XCCL,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L XCCL C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
7,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,C VIEW,XCCL,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
8,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,SecurView Secondary Capture,74100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
9,4330018595,1965-07-01,54.0,60103700,2020-06-02,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
